# FIFA World Cup 2026 — Analytic Tasks
**Raw data:** `World_Cup_2026.xlsx` (Group Stage + Knockouts sheets)
**Clean dataset produced below:** `World_Cup_2026_clean.xlsx` (`Matches` + `TeamMatch` sheets)

This notebook performs **four distinct analytic tasks**, each driven by its own question and
using all six required skills: analytic question formulation, data wrangling, data preparation
and sampling, descriptive statistics, a confidence interval, and a t-test (one-sample or
two-sample).

**Structure:** Section 0 does the one-time cleaning/wrangling of the messy raw workbook and
saves a tidy, standardized dataset to disk (`World_Cup_2026_clean.xlsx`). Every task below is
then fully self-contained: it **re-reads that clean file from scratch** rather than reusing
variables left over from another cell — this is the more standard way to structure an analysis
so each task can be run independently, in any order, or handed to someone else.


## 0. One-Time Data Wrangling → Save Clean Dataset

The raw workbook has two sheets with inconsistent layouts:
- **Group Stage** — one clean header row, 72 matches.
- **Knockouts** — has round-marker rows (`ROUND OF 32`, `ROUND OF 16`, ...) and a *repeated*
  header row before every round, plus combined `Yellow Cards`/`Red Cards` columns
  (e.g. `"0Y/2Y"`) instead of separate Team 1 / Team 2 columns.

This cell cleans both sheets, aligns their columns, combines them, builds a long
"team-match" table (one row per team per match — needed for group comparisons later), and
**writes both tables to a new Excel file, `World_Cup_2026_clean.xlsx`**, with one sheet each.
This only needs to run once.


In [2]:
import pandas as pd
import numpy as np
import re
from scipy import stats

RAW_PATH   = "World_Cup_2026.xlsx"        # original raw workbook
CLEAN_PATH = "World_Cup_2026_clean.xlsx"  # tidy dataset we build and save here

# ---- Group Stage ----
gs_raw = pd.read_excel(RAW_PATH, sheet_name="Group Stage", header=1).dropna(how="all")
gs_raw["Stage"] = "Group Stage"

# ---- Knockouts: strip round-marker rows, forward-fill the round, rebuild header ----
ko_raw = pd.read_excel(RAW_PATH, sheet_name="Knockouts", header=None)
current_round, header, rows = None, None, []
for i in range(ko_raw.shape[0]):
    row = ko_raw.iloc[i]
    if row.notna().sum() == 0:
        continue
    if row.notna().sum() == 1 and pd.isna(row.iloc[1]):
        current_round = row.iloc[0]          # e.g. "ROUND OF 32"
        continue
    if row.iloc[0] == "Date":
        header = row.tolist()                # repeated header row
        continue
    rows.append([current_round] + row.tolist())

ko = pd.DataFrame(rows, columns=["Round"] + header)
ko["Stage"] = "Knockout"

# split "0Y/2Y" -> Yellow T1, Yellow T2 (and similarly for Red Cards)
def split_cards(val):
    if pd.isna(val):
        return (np.nan, np.nan)
    parts = str(val).split("/")
    nums = [int(re.sub(r"[^0-9]", "", p)) for p in parts]
    return nums[0], nums[1]

ko[["Yellow T1", "Yellow T2"]] = ko["Yellow Cards"].apply(lambda v: pd.Series(split_cards(v)))
ko[["Red T1", "Red T2"]] = ko["Red Cards"].apply(lambda v: pd.Series(split_cards(v)))
ko = ko.drop(columns=["Yellow Cards", "Red Cards"])

# clean score strings like "1 (p)" (won on pens) / "p0" (lost on pens, 0 goals)
def clean_score(val):
    if pd.isna(val):
        return np.nan
    digits = re.findall(r"\d+", str(val))
    return int(digits[0]) if digits else np.nan

for df in (gs_raw, ko):
    df["Score 1"] = df["Score 1"].apply(clean_score)
    df["Score 2"] = df["Score 2"].apply(clean_score)

common_cols = ["Date","Stage","Team 1","Team 2","Score 1","Score 2",
               "Shots T1","Shots T2","Shots On Target T1","Shots On Target T2",
               "Match Corners T1","Match Corners T2","Fouls T1","Fouls T2",
               "xG T1","xG T2","Yellow T1","Yellow T2","Red T1","Red T2"]

matches = pd.concat([gs_raw[common_cols], ko[common_cols]], ignore_index=True)
matches["MatchID"] = matches.index
print("Total matches:", matches.shape[0])
print(matches["Stage"].value_counts())

# ---- Build the long team-match table (one row per team per match) ----
def to_long(df):
    t1 = pd.DataFrame({
        "MatchID": df["MatchID"], "Stage": df["Stage"], "Team": df["Team 1"], "Opponent": df["Team 2"],
        "GoalsFor": df["Score 1"], "GoalsAgainst": df["Score 2"],
        "Shots": df["Shots T1"], "SOT": df["Shots On Target T1"],
        "Corners": df["Match Corners T1"], "Fouls": df["Fouls T1"], "xG": df["xG T1"],
    })
    t2 = pd.DataFrame({
        "MatchID": df["MatchID"], "Stage": df["Stage"], "Team": df["Team 2"], "Opponent": df["Team 1"],
        "GoalsFor": df["Score 2"], "GoalsAgainst": df["Score 1"],
        "Shots": df["Shots T2"], "SOT": df["Shots On Target T2"],
        "Corners": df["Match Corners T2"], "Fouls": df["Fouls T2"], "xG": df["xG T2"],
    })
    long_df = pd.concat([t1, t2], ignore_index=True)
    long_df["Result"] = np.select(
        [long_df["GoalsFor"] > long_df["GoalsAgainst"], long_df["GoalsFor"] < long_df["GoalsAgainst"]],
        ["Win", "Loss"], default="Draw")
    return long_df

team_match = to_long(matches)

# ---- Save the tidy dataset to its own Excel file, ready for every task to fetch ----
with pd.ExcelWriter(CLEAN_PATH, engine="openpyxl") as writer:
    matches.to_excel(writer, sheet_name="Matches", index=False)
    team_match.to_excel(writer, sheet_name="TeamMatch", index=False)

print(f"Saved clean dataset to: {CLEAN_PATH}")
print("Matches sheet:   ", matches.shape, "-> Stage counts:", dict(matches['Stage'].value_counts()))
print("TeamMatch sheet: ", team_match.shape, "-> Result counts:", dict(team_match['Result'].value_counts()))



Total matches: 103
Stage
Group Stage    72
Knockout       31
Name: count, dtype: int64
Saved clean dataset to: World_Cup_2026_clean.xlsx
Matches sheet:    (103, 21) -> Stage counts: {'Group Stage': np.int64(72), 'Knockout': np.int64(31)}
TeamMatch sheet:  (206, 12) -> Result counts: {'Win': np.int64(79), 'Loss': np.int64(79), 'Draw': np.int64(48)}


We also reshape the match-level table into a **team-match (long) table**, giving one row
per team per match. This is the population several of our tasks (2, 3 and 4) will sample from,
since questions like *"do winning teams get more corners?"* are really about **teams**, not matches.


In [ ]:
# ---- Build the long team-match table (one row per team per match) ----
def to_long(df):
    t1 = pd.DataFrame({
        "MatchID": df["MatchID"], "Stage": df["Stage"], "Team": df["Team 1"], "Opponent": df["Team 2"],
        "GoalsFor": df["Score 1"], "GoalsAgainst": df["Score 2"],
        "Shots": df["Shots T1"], "SOT": df["Shots On Target T1"],
        "Corners": df["Match Corners T1"], "Fouls": df["Fouls T1"], "xG": df["xG T1"],
    })
    t2 = pd.DataFrame({
        "MatchID": df["MatchID"], "Stage": df["Stage"], "Team": df["Team 2"], "Opponent": df["Team 1"],
        "GoalsFor": df["Score 2"], "GoalsAgainst": df["Score 1"],
        "Shots": df["Shots T2"], "SOT": df["Shots On Target T2"],
        "Corners": df["Match Corners T2"], "Fouls": df["Fouls T2"], "xG": df["xG T2"],
    })
    long_df = pd.concat([t1, t2], ignore_index=True)
    long_df["Result"] = np.select(
        [long_df["GoalsFor"] > long_df["GoalsAgainst"], long_df["GoalsFor"] < long_df["GoalsAgainst"]],
        ["Win", "Loss"], default="Draw")
    return long_df

team_match = to_long(matches)

# ---- Save the tidy dataset to its own Excel file, ready for every task to fetch ----
with pd.ExcelWriter(CLEAN_PATH, engine="openpyxl") as writer:
    matches.to_excel(writer, sheet_name="Matches", index=False)
    team_match.to_excel(writer, sheet_name="TeamMatch", index=False)

print(f"Saved clean dataset to: {CLEAN_PATH}")
print("Matches sheet:   ", matches.shape, "-> Stage counts:", dict(matches['Stage'].value_counts()))
print("TeamMatch sheet: ", team_match.shape, "-> Result counts:", dict(team_match['Result'].value_counts()))


From this point on, **every task reads fresh from `World_Cup_2026_clean.xlsx`** using
`pd.read_excel(CLEAN_PATH, sheet_name=...)` — none of them depend on the `matches` / `team_match`
variables above still being in memory.
